# F5-TTS Pipeline Worker - Colab GPU

Runs the **F5-TTS worker** on Colab's free GPU (T4). Unlike the Abogen worker,
this notebook does **not** need a separate Flask server - the worker synthesizes
audio directly using F5-TTS zero-shot voice cloning.

## Before you start

On your local machine, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# copy the tcp://X.tcp.ngrok.io:XXXXX URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

When you fill `REDIS_URL` later:
- Use `redis://host:port` for plain TCP tunnels (typical ngrok TCP endpoint)
- Use `rediss://host:port` only when your Redis endpoint requires TLS

Set Colab Secrets (key icon, left sidebar):

| Secret | Value |
|--------|-------|
| `SSH_HOST` | server IP / hostname (optional) |
| `SSH_USER` | SSH username (optional) |
| `SSH_KEY`  | full private key (optional) |
| `SSH_REMOTE_PATH` | remote output dir, e.g. `/opt/tts-node/outputs` |
| `SSH_PORT` | SSH port (default 22) |
| `GITHUB_TOKEN` | PAT for private repos (optional) |
| `GITHUB_USER`  | GitHub username (optional) |

If SSH secrets are omitted, output stays on Google Drive.

Runtime -> Change runtime type -> T4 GPU before running.


In [ ]:
# 1. Mount Google Drive (fallback output + reference audio storage)
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
os.makedirs('/content/ref', exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# -- 2. Secrets ---------------------------------------------------------------
# Add these in the Secrets panel (key icon, left sidebar):
#
#   GITHUB_TOKEN      -> ghp_...  (Personal Access Token - private repos only)
#   GITHUB_USER       -> your GitHub username
#
#   SSH_HOST          -> server IP or hostname (optional)
#   SSH_USER          -> SSH username           (optional)
#   SSH_KEY           -> private key contents   (optional)
#   SSH_REMOTE_PATH   -> remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          -> SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'SSH host: {"ok" if SSH_HOST else "not set (optional)"}')
print(f'SSH key:  {"ok" if SSH_KEY else "not set (optional)"}')


In [ ]:
# -- 3. Optional SSH setup + mount remote output dir via sshfs -----------------
import os, subprocess, textwrap, re

if SSH_HOST and SSH_USER and SSH_KEY:
    SSH_KEY_PATH = '/root/.ssh/colab_key'
    os.makedirs('/root/.ssh', exist_ok=True)

    def _reconstruct_pem(raw: str) -> str:
        key = raw.replace('\\n', '\n').strip()
        pattern = r'(-----BEGIN [^-]+-----)(.*?)(-----END [^-]+-----)'
        match = re.search(pattern, key, re.DOTALL)
        if match:
            header, body, footer = match.groups()
            body = ''.join(body.split())
            wrapped = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
            return f'{header}\n{wrapped}\n{footer}\n'
        return key + '\n'

    with open(SSH_KEY_PATH, 'w') as f:
        f.write(_reconstruct_pem(SSH_KEY))
    os.chmod(SSH_KEY_PATH, 0o600)

    # Validate key format
    check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
    if check.returncode != 0:
        print('Debug - First 100 chars of processed key:\n', open(SSH_KEY_PATH).read()[:100])
        raise RuntimeError(f'SSH Key Error: {check.stderr}')

    # Configure SSH client
    with open('/root/.ssh/config', 'w') as f:
        f.write(textwrap.dedent(f"""\
            Host {SSH_HOST}
                StrictHostKeyChecking no
                UserKnownHostsFile /dev/null
                IdentityFile {SSH_KEY_PATH}
                Port {SSH_PORT}
                ConnectTimeout 10
        """))

    print(f'SSH key verified. Testing connection to {SSH_USER}@{SSH_HOST}...')

    # Test connectivity and ensure remote path exists
    test_conn = subprocess.run(['ssh', f'{SSH_USER}@{SSH_HOST}', f'mkdir -p {SSH_REMOTE_PATH}'], capture_output=True, text=True)
    if test_conn.returncode != 0:
        raise RuntimeError(f'Connection failed: {test_conn.stderr}')

    # Mount via sshfs
    !apt-get install -y sshfs > /dev/null 2>&1
    SSHFS_MOUNT = '/content/remote-outputs'
    os.makedirs(SSHFS_MOUNT, exist_ok=True)
    subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

    mount_cmd = [
        'sshfs', f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}', SSHFS_MOUNT,
        '-o', f'IdentityFile={SSH_KEY_PATH},port={SSH_PORT}',
        '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3'
    ]

    result = subprocess.run(mount_cmd, capture_output=True, text=True)
    if result.returncode == 0:
        OUTPUT_DIR = SSHFS_MOUNT
        os.environ['OUTPUT_DIR'] = OUTPUT_DIR
        print(f'Successfully mounted {SSH_REMOTE_PATH} to {SSHFS_MOUNT}')
    else:
        print(f'Mount failed: {result.stderr}')
        print('Continuing with Google Drive output directory.')
else:
    print('SSH secrets not provided - using Google Drive output only.')


In [ ]:
# -- 4. Config - fill these in --------------------------------------------------
# Use redis:// for plain TCP tunnels (ngrok/cloudflared).
# Use rediss:// only when the endpoint requires TLS.
REDIS_URL = 'rediss://redis.0xvictor.dev:6380'
WORKER_ID = 'colab-f5-1'

# Reference audio settings.
# If you provide your own WAV in Drive, set F5_REF_TEXT to the exact transcript
# of that WAV. Mismatched text can cause unstable pacing/timbre.
F5_REF_AUDIO = '/content/ref/reference.wav'
F5_REF_TEXT  = 'the invention of movable metal letters in the middle of the fifteenth century may justly be considered as the invention of the art of printing'

# Base narration speed (lower = slower). If speech sounds too fast, reduce this.
F5_SPEED = 0.90

os.environ['REDIS_URL']       = REDIS_URL
os.environ['WORKER_ID']       = WORKER_ID
os.environ['OUTPUT_DIR']      = OUTPUT_DIR
os.environ['SPOOL_DIR']       = SPOOL_DIR
os.environ['USE_GPU']         = 'true'
os.environ['F5_DEVICE']       = 'cuda'
os.environ['F5_MODEL']        = 'F5TTS_v1_Base'
os.environ['F5_REF_AUDIO']    = F5_REF_AUDIO
os.environ['F5_REF_TEXT']     = F5_REF_TEXT
os.environ['F5_SPEED']        = str(F5_SPEED)
os.environ['F5_MIN_SPEED']    = '0.6'
os.environ['F5_MAX_SPEED']    = '1.3'
# Keep segments short enough that ref_mel + gen_mel stays within the model's
# positional-encoding budget.  The original 600 then 400 caused systematic tensor
# mismatches when the reference audio exceeded ~10 s.
os.environ['F5_MAX_CHARS']    = '200'
os.environ['F5_PROGRESS_LOG_STEP'] = '10'
os.environ['PROMETHEUS_PORT'] = '8000'
# Rolling context: use last N seconds of generated audio as ref for next segment.
os.environ['F5_ROLLING_CTX']       = 'true'
os.environ['F5_ROLLING_CTX_SECS']  = '3.0'
# Chapter / heading padding (silence + crossfade) applied to detected headings.
os.environ['F5_CHAPTER_PAUSE_PRE_MS']  = '1500'
os.environ['F5_CHAPTER_PAUSE_POST_MS'] = '2500'
os.environ['F5_CHAPTER_FADE_MS']       = '400'
# Allow recursive retry to split down to ~60-char micro-segments as a last
# resort (extra insurance if the reference is still borderline-long).
os.environ['F5_RETRY_SPLIT_MIN_CHARS'] = '30'
os.environ['F5_RETRY_SPLIT_MAX_DEPTH'] = '6'

print('Config set.')


In [ ]:
# 5. Install system dependencies
!apt-get install -y ffmpeg > /dev/null 2>&1
print('ffmpeg installed.')

In [ ]:
# 6. Install F5-TTS + GPU PyTorch
# Colab T4 runs CUDA 12.x â€” use the cu121 wheel
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q f5-tts soundfile pydub pysbd redis prometheus_client pyngrok
print('All packages installed.')

In [ ]:
# -- 7. Reference audio ----------------------------------------------------------
# For novel narration, use a real narrator-style WAV (10-30s of clean speech).
# If using your own WAV, F5_REF_TEXT must exactly match what is spoken.
# Mismatched transcript is a common reason for overly fast/odd output.

import os, shutil, glob

DRIVE_REF = '/content/drive/MyDrive/tts-pipeline/reference.wav'

if os.path.exists(DRIVE_REF):
    shutil.copy(DRIVE_REF, F5_REF_AUDIO)
    ref_text = (os.environ.get('F5_REF_TEXT') or '').strip()
    if not ref_text:
        raise RuntimeError(
            'Drive reference detected, but F5_REF_TEXT is empty. '\
            'Set F5_REF_TEXT in cell 4 to the exact words spoken in reference.wav.'
        )
    os.environ['F5_REF_AUDIO'] = F5_REF_AUDIO
    os.environ['F5_REF_TEXT'] = ref_text
    print(f'Loaded narrator reference from Drive ({os.path.getsize(F5_REF_AUDIO):,} bytes)')
    print(f'Ref text: "{ref_text}"')
else:
    # Fallback to bundled package sample (works without custom transcript)
    try:
        import f5_tts
        pkg_root = os.path.dirname(f5_tts.__file__)
        candidates = glob.glob(os.path.join(pkg_root, '**', 'basic_ref_en.wav'), recursive=True)
        if not candidates:
            raise RuntimeError('No bundled basic_ref_en.wav found in f5_tts package')

        shutil.copy(candidates[0], F5_REF_AUDIO)
        fallback_text = 'Some call me nature, others call me mother nature.'
        os.environ['F5_REF_AUDIO'] = F5_REF_AUDIO
        os.environ['F5_REF_TEXT'] = fallback_text
        print('Drive reference not found - using bundled sample (quality may vary for novels)')
        print('Tip: upload a narrator WAV to Drive -> My Drive/tts-pipeline/reference.wav')
    except Exception as e:
        print(f'ERROR loading bundled sample: {e}')

if not os.path.exists(F5_REF_AUDIO):
    raise RuntimeError(f'No reference audio at {F5_REF_AUDIO} - worker cannot start.')


In [ ]:
# -- 7b. Trim reference audio to ≤12 s -----------------------------------------
# Root cause of the "Sizes of tensors must match except in dimension 2" errors:
# F5-TTS concatenates [ref_mel, gen_mel] and the joint sequence must fit within
# the model's positional-encoding budget.  A ~20 s reference (~1930 mel frames)
# leaves too little headroom for generated audio, causing mismatches on nearly
# every chunk.  Trimming to 8 s (~750 frames) gives enough headroom for generated audio.
import subprocess, os

MAX_REF_SECS = 8
ref_src  = os.environ.get('F5_REF_AUDIO', '/content/ref/reference.wav')
ref_trim = ref_src.replace('.wav', '_trimmed.wav')

probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'default=noprint_wrappers=1:nokey=1', ref_src],
    capture_output=True, text=True,
)
duration = float(probe.stdout.strip() or '0')
print(f'Reference duration: {duration:.2f}s')

if duration > MAX_REF_SECS:
    subprocess.run(
        ['ffmpeg', '-y', '-i', ref_src,
         '-t', str(MAX_REF_SECS), '-ar', '24000', '-ac', '1', ref_trim],
        check=True, capture_output=True,
    )
    os.environ['F5_REF_AUDIO'] = ref_trim
    F5_REF_AUDIO = ref_trim
    print(f'Trimmed to {MAX_REF_SECS}s -> {ref_trim}')
else:
    os.environ['F5_REF_AUDIO'] = ref_src
    print(f'Reference is {duration:.2f}s (≤{MAX_REF_SECS}s) — no trim needed.')


In [ ]:
import os, subprocess, shutil

REPO_URL = 'https://github.com/wizardgang/audiobook-stack.git'
REPO_DIR = '/content/audiobook-stack'
WORKER_SCRIPT_PATH_IN_REPO = 'f5-worker/worker.py'
DEST_WORKER_PATH = '/content/worker.py'

print(f'Cloning repository {REPO_URL}...')
# Remove existing repo if any, to ensure clean clone
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
print('Repository cloned.')

# # --- Debugging step: List repository contents to verify path ---
# print(f'Listing contents of {REPO_DIR}:')
# result = subprocess.run(['ls', '-R', REPO_DIR], capture_output=True, text=True, check=True)
# print(result.stdout)
# # --------------------------------------------------------------

# Copy the worker script
shutil.copy(os.path.join(REPO_DIR, WORKER_SCRIPT_PATH_IN_REPO), DEST_WORKER_PATH)
print(f'worker.py ready from {WORKER_SCRIPT_PATH_IN_REPO} in cloned repo.')

# Optional: Clean up cloned repository to save space
# shutil.rmtree(REPO_DIR)
# print('Cloned repository removed.')

In [ ]:
# -- 9b. Inspect and optionally reset stuck book counters ----------------------
# If a book's done_chunks counter is above total_chunks from a previous failed
# run, the merger fires immediately on the first new chunk and the book is
# never fully synthesised.  Inspect here before re-queuing.
import redis as _redis_mod, os

_r2 = _redis_mod.from_url(os.environ['REDIS_URL'], decode_responses=True, socket_connect_timeout=5)

book_keys = sorted(_r2.keys('book:*'))
if not book_keys:
    print('No books in Redis.')
else:
    print(f'{"Key":<25} {"done":>6} {"total":>6}  status')
    print('-' * 55)
    for k in book_keys:
        info  = _r2.hgetall(k)
        done  = int(info.get('done_chunks', 0))
        total = int(info.get('total_chunks', 0))
        flag  = ' ← OVER-COUNT' if total and done > total else ''
        print(f'{k:<25} {done:>6} {total:>6}  {info.get("status","?")}{flag}')

# To reset a stuck book, uncomment and fill in the book_id, then re-run:
# RESET_BOOK_ID = '38064858'
# _r2.hset(f'book:{RESET_BOOK_ID}', mapping={'done_chunks': '0', 'status': 'pending'})
# print(f'Reset done_chunks for book:{RESET_BOOK_ID}')


In [ ]:
#9. Verify Redis + output directory
import redis, os
from pathlib import Path

# Redis
r = redis.from_url(os.environ['REDIS_URL'], decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

# Output dir write test
out = Path(os.environ['OUTPUT_DIR'])
out.mkdir(parents=True, exist_ok=True)
probe = out / '.write_test'
try:
    probe.touch()
    probe.unlink()
    print(f'Output dir writable:  {out} ')
except OSError as e:
    print(f'Output dir NOT writable: {out}  - ({e})')
    print('  â†’ chunks will spool to', os.environ['SPOOL_DIR'])

# Spool check
spool = Path(os.environ['SPOOL_DIR'])
spooled = list(spool.rglob('*.mp3'))
if spooled:
    print(f'WARNING: {len(spooled)} MP3(s) stuck in spool from a previous run:')
    for f in spooled[:5]:
        print(f'  {f}')
else:
    print(f'Spool clean: {spool}')

In [ ]:
# -- 10. Warm up F5-TTS model (downloads weights ~1.2 GB on first run) ----------
print('Loading F5-TTS model... (first run downloads ~1.2 GB, subsequent runs are instant)')
import sys
sys.path.insert(0, '/content')

class _NoopProgress:
    def __call__(self, *args, **kwargs):
        pass
    def tqdm(self, iterable, *args, **kwargs):
        return iterable

_noop = lambda *args, **kwargs: None

from f5_tts.api import F5TTS
import os
speed = float(os.environ.get('F5_SPEED', '0.9'))
_engine = F5TTS(
    model=os.environ['F5_MODEL'],
    device=os.environ['F5_DEVICE'],
)
print(f'Model loaded. Running test synthesis at speed={speed:.2f}...')

import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    wav, sr, _ = _engine.infer(
        ref_file=os.environ['F5_REF_AUDIO'],
        ref_text=os.environ['F5_REF_TEXT'],
        gen_text='Those sweatpants? she volleys back, pointing at the tie-dyed pair that match hers.They’re a keepsake, I say defensively, curling my legs under me.',
        speed=speed,
        show_info=_noop,
        progress=_NoopProgress(),
    )
print(f'Smoke test OK - generated {len(wav)/sr:.2f}s of audio at {sr} Hz')
print('If this still sounds too fast, reduce F5_SPEED in cell 4 (for example 0.8) and re-run.')


In [ ]:
# -- 11. Save smoke-test sample audio to Drive ----------------------------------
# Run this after cell 10. It saves both WAV and MP3 so you can review pacing quickly.
import os, time, io
import soundfile as sf
from pydub import AudioSegment

if 'wav' not in globals() or 'sr' not in globals():
    raise RuntimeError('Smoke-test audio not found in memory. Run cell 10 first.')

samples_dir = '/content/drive/MyDrive/tts-pipeline/samples'
os.makedirs(samples_dir, exist_ok=True)
stamp = time.strftime('%Y%m%d_%H%M%S')
base = f'f5_smoke_{stamp}'
wav_path = os.path.join(samples_dir, base + '.wav')
mp3_path = os.path.join(samples_dir, base + '.mp3')

# Save WAV
sf.write(wav_path, wav, sr, subtype='PCM_16')

# Save MP3
buf = io.BytesIO()
sf.write(buf, wav, sr, format='WAV', subtype='PCM_16')
buf.seek(0)
AudioSegment.from_wav(buf).export(mp3_path, format='mp3', bitrate='128k')

print(f'Saved smoke sample WAV: {wav_path}')
print(f'Saved smoke sample MP3: {mp3_path}')


In [ ]:
import os
from pyngrok import ngrok, conf
import subprocess

# Define the ngrok path
NGROK_BIN_PATH = '/usr/local/bin/ngrok'
NGROK_ZIP_PATH = 'ngrok-v3-stable-linux-amd64.tgz'
NGROK_DOWNLOAD_URL = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'

print(f"Checking for ngrok at {NGROK_BIN_PATH}...")

# Ensure ngrok is correctly installed manually
try:
    # Clean up previous download attempts and any existing binary
    subprocess.run(['rm', '-f', NGROK_ZIP_PATH, 'ngrok', NGROK_BIN_PATH], check=False, capture_output=True)

    print("Attempting to install ngrok manually.")

    # Download ngrok using curl for better compatibility
    print(f"Downloading ngrok from {NGROK_DOWNLOAD_URL}...")
    download_result = subprocess.run(['curl', '-L', '-o', NGROK_ZIP_PATH, NGROK_DOWNLOAD_URL], check=True, capture_output=True)
    if download_result.returncode != 0:
        raise RuntimeError(f"curl failed: {download_result.stderr.decode()}")
    if not os.path.exists(NGROK_ZIP_PATH):
        raise FileNotFoundError(f"{NGROK_ZIP_PATH} not found after curl.")
    print("Download complete.")

    # Unzip ngrok (using tar for .tgz files)
    print(f"Extracting {NGROK_ZIP_PATH}...")
    # Using tar for .tgz files instead of unzip
    extract_result = subprocess.run(['tar', '-xzf', NGROK_ZIP_PATH], check=True, capture_output=True)
    if extract_result.returncode != 0:
        raise RuntimeError(f"tar failed: {extract_result.stderr.decode()}")
    if not os.path.exists('ngrok'):
        raise FileNotFoundError("'ngrok' executable not found after extraction.")
    print("Extraction complete.")

    # Move and make executable
    print(f"Moving ngrok to {NGROK_BIN_PATH} and setting permissions...")
    mv_result = subprocess.run(['mv', 'ngrok', NGROK_BIN_PATH], check=True, capture_output=True)
    if mv_result.returncode != 0:
        raise RuntimeError(f"mv failed: {mv_result.stderr.decode()}")
    chmod_result = subprocess.run(['chmod', '+x', NGROK_BIN_PATH], check=True, capture_output=True)
    if chmod_result.returncode != 0:
        raise RuntimeError(f"chmod failed: {chmod_result.stderr.decode()}")
    print(f"ngrok installed manually to {NGROK_BIN_PATH}.")

    # Verify ngrok installation
    print("Verifying ngrok installation...")
    version_output = subprocess.run([NGROK_BIN_PATH, "version"], check=True, capture_output=True, text=True, timeout=5)
    print("ngrok verified successfully:")
    print(version_output.stdout.strip())
    # Configure pyngrok to use the manually installed ngrok binary path.
    conf.get_default().ngrok_path = NGROK_BIN_PATH
    print(f"Pyngrok configured to use ngrok binary at: {conf.get_default().ngrok_path}")

except Exception as e:
    print(f"CRITICAL ERROR: ngrok manual installation failed: {e}")
    print("Pyngrok will attempt to use its default installation method (which may fail with 403).")
    conf.get_default().ngrok_path = None # Revert to default, letting pyngrok try (and likely fail)

prom_port = int(os.environ.get('PROMETHEUS_PORT', '8000'))

if not NGROK_TOKEN:
    print('NGROK_TOKEN not set - skipping Prometheus tunnel.')
    print(f'Local metrics endpoint: http://127.0.0.1:{prom_port}/metrics')
else:
    try:
        # pyngrok.set_auth_token will implicitly call install_ngrok if pyngrok.conf.DEFAULT_NGROK_PATH is not set
        # or if the binary at that path is invalid. By setting it explicitly above, we aim to bypass this.
        ngrok.set_auth_token(NGROK_TOKEN)
        # reset any old tunnels in this runtime so the metrics URL is unambiguous
        ngrok.kill()
        tunnel = ngrok.connect(addr=prom_port, proto='http')
        os.environ['PROMETHEUS_PUBLIC_URL'] = tunnel.public_url
        print('Prometheus tunnel live:')
        print(f'  {tunnel.public_url}/metrics')
    except Exception as e:
        print(f"ERROR: Could not establish ngrok tunnel: {e}")
        print(f"Ensure your NGROK_TOKEN is valid and ngrok is properly installed.")

In [ ]:
# -- 12. Start the F5-TTS worker
# Runs until session ends or interrupted. Pulls chunk jobs from pipeline:tts,
# writes MP3s to OUTPUT_DIR (or spools locally if the mount is offline).
%run /content/worker.py
